# Worked example — no code writing
**Run Runtime → Run all**, or read the saved example outputs below. Each step is explained before its code cell.

[Get an API key](https://github.com/manueltonneau/hpi-api/blob/main/SETUP.md) · [Request parameters](https://github.com/manueltonneau/hpi-api/blob/main/PARAMETERS.md)

## 1. Choose settings
The next cell imports libraries and sets the collection options using Colab form controls.

**Your action:** keep **USE_SAVED_EXAMPLE** checked for Juan’s ten saved AfD videos. For live data, uncheck it, enter your chosen **CHANNEL_HANDLE** (such as `@GoogleDevelopers`), and enter your key in the hidden prompt. **ORDER** chooses views or publication date; **NUMBER_OF_VIDEOS** sets the page size (1–20).

**Output:** the selected mode. After changing settings, rerun all cells below. Saved mode only sorts the ten historical records; it cannot obtain another channel or current counts.


In [1]:
# @title 1. Choose settings { display-mode: "form" }
import os
import json
import time
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone
import pandas as pd

USE_SAVED_EXAMPLE = True  # @param {type:"boolean"}
CHANNEL_HANDLE = "@GoogleDevelopers"  # @param {type:"string"}
ORDER = "viewCount"  # @param ["viewCount", "date"]
NUMBER_OF_VIDEOS = 10  # @param {type:"integer"}

OFFLINE = USE_SAVED_EXAMPLE
CHANNEL_ID = "UC_dZp8bZipnjntBGLVHm6rw"  # Saved example only; resolved from handle in live mode.
PAGE_SIZE = NUMBER_OF_VIDEOS
BASE_ORDER = ORDER
OUTPUT_DIR = Path("workshop_outputs")
if not 1 <= PAGE_SIZE <= 20:
    raise ValueError("Choose 1–20 videos.")
API_KEY = os.environ.get("YOUTUBE_API_KEY", "")
if not OFFLINE and not API_KEY:
    API_KEY = getpass("YouTube API key (hidden): ").strip()
if not OFFLINE and not API_KEY:
    raise ValueError("Enter a key, or use the saved example.")
print("Saved example" if OFFLINE else "Live collection")


Saved example


## 2. Load the fallback data
This cell loads ten historical saved records and defines simulated responses for saved mode.

**Your action:** run unchanged in either mode. **Output:** “Loaded 10 saved records; no network requests.” Live mode does not use these records for collection. The fallback responses are reconstructed from Juan’s notebook outputs; their original observation time is unknown.


In [2]:
# @title 2. Load saved example { display-mode: "form" }
FIXTURE = {'provenance': {'description': 'Normalized records reconstructed from displayed saved outputs, not raw API responses. Original exact observation time unknown. Video publication dates are not collection timestamps.', 'source_file': 'youtube_germany_annotated2.ipynb', 'source_cells_zero_based': [12, 14], 'records': 10, 'scope': 'Ten videos printed by the supplied notebook. Coverage and original search completeness have not been verified.'}, 'records': [{'video_id': '8ioWqF4k2fo', 'title': 'Brandmauer weg oder Neuwahlen! PK mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag', 'published_at': '2026-05-19T12:58:56Z', 'duration': 'PT23M7S', 'views': 308065, 'likes': 25574, 'comments': 2288, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'eYh7yfnluqs', 'title': 'Interview mit Beatrix von Storch zum Ergebnis des Koalitionsgipfels! - AfD-Fraktion im Bundestag', 'published_at': '2026-05-13T15:01:09Z', 'duration': 'PT17M35S', 'views': 249565, 'likes': 20505, 'comments': 1563, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'Uf9nmuTDesk', 'title': 'Regierungsbefragung mit Lars Klingbeil und Verena Hubertz! - AfD-Fraktion im Bundestag', 'published_at': '2026-05-20T14:50:16Z', 'duration': 'PT17M9S', 'views': 149674, 'likes': 7771, 'comments': 1822, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'Hz6SLP23Nv0', 'title': 'LIVE: Pressekonferenz der AfD-Fraktion - Diese Woche im Bundestag', 'published_at': '2026-05-19T08:26:19Z', 'duration': 'PT1H9M57S', 'views': 147981, 'likes': 11521, 'comments': 1183, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'OU-pD4Tm90w', 'title': 'BUNDESTAG LIVE - 80. Sitzung - AfD-Fraktion im Bundestag', 'published_at': '2026-05-21T21:23:45Z', 'duration': 'PT11H55M1S', 'views': 117253, 'likes': 2224, 'comments': 5, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'HbPnlbTchM8', 'title': 'Dr. Anna Rathert zum Untersuchungsausschuss Nord-Stream! - AfD-Fraktion im Deutschen Bundestag', 'published_at': '2026-05-21T10:47:14Z', 'duration': 'PT4M32S', 'views': 112037, 'likes': 7779, 'comments': 805, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': '5m7dIsDftmQ', 'title': 'Bernd Baumann knöpft sich die Umfrageverlierer Söder, Maier und Banaszak vor! - AfD-Fraktion', 'published_at': '2026-05-19T09:17:11Z', 'duration': 'PT2M22S', 'views': 103756, 'likes': 7568, 'comments': 299, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'IehYwnYBwZs', 'title': 'BUNDESTAG LIVE - 79. Sitzung - AfD-Fraktion im Bundestag', 'published_at': '2026-05-20T18:51:36Z', 'duration': 'PT6H41M30S', 'views': 98033, 'likes': 2274, 'comments': 25, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': '40qjQvDzUII', 'title': 'Nordstream: Wir fordern einen Untersuchungsausschuss! Markus Frohnmaier im Interview - AfD-Fraktion', 'published_at': '2026-05-20T16:00:05Z', 'duration': 'PT12M53S', 'views': 81091, 'likes': 7003, 'comments': 695, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}, {'video_id': 'iwycKryn3aw', 'title': 'LIVE: Presseerklärung mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag', 'published_at': '2026-05-19T12:58:22Z', 'duration': 'PT23M32S', 'views': 72894, 'likes': 6037, 'comments': 612, 'channel_id': 'UC_dZp8bZipnjntBGLVHm6rw', 'source_kind': 'saved_notebook_output', 'observed_at': None}]}

OFFLINE_PAGE = "offline-page:"

def offline_payload(endpoint, parameters):
    if CHANNEL_ID != "UC_dZp8bZipnjntBGLVHm6rw":
        raise ValueError("Offline data contain only Juan's example channel. Use live mode for another channel.")
    records = FIXTURE["records"]
    if endpoint == "search":
        if parameters.get("channelId") != CHANNEL_ID:
            raise ValueError("The offline cache cannot answer a request for another channel.")
        if parameters.get("type") != "video" or parameters.get("q"):
            raise ValueError("Offline fixtures support video searches without keywords only.")
        order = parameters.get("order", "viewCount")
        if order not in {"date", "viewCount"}:
            raise ValueError("Offline exercise supports date or viewCount ordering.")
        field = "published_at" if order == "date" else "views"
        ranked = sorted(records, key=lambda r: r[field], reverse=True)
        token = parameters.get("pageToken", OFFLINE_PAGE + "0")
        if not token.startswith(OFFLINE_PAGE):
            raise ValueError("Unexpected offline page token.")
        start, size = int(token[len(OFFLINE_PAGE):]), int(parameters["maxResults"])
        selected = ranked[start:start + size]
        payload = {"items": [{"id": {"videoId": r["video_id"]},
                              "snippet": {"title": r["title"]}} for r in selected]}
        if start + size < len(ranked):
            payload["nextPageToken"] = OFFLINE_PAGE + str(start + size)
        return payload
    if endpoint == "videos":
        wanted = parameters.get("id", "").split(",")
        by_id = {r["video_id"]: r for r in records}
        return {"items": [{"id": v,
            "snippet": {"title": by_id[v]["title"], "publishedAt": by_id[v]["published_at"],
                        "channelId": by_id[v]["channel_id"]},
            "statistics": {api: str(by_id[v][local]) for api, local in
                           [("viewCount", "views"), ("likeCount", "likes"), ("commentCount", "comments")]
                           if by_id[v][local] is not None},
            "contentDetails": {"duration": by_id[v]["duration"]}}
            for v in wanted if v in by_id]}
    raise ValueError("This endpoint is not available in offline fixtures.")

print("Loaded", len(FIXTURE["records"]), "saved records; no network requests.")


Loaded 10 saved records; no network requests.


## 3. Prepare requests
This cell defines `api_get()`: a helper that sends a GET request, adds your key, checks errors and returns JSON as a Python dictionary. In saved mode it uses the fallback instead.

**Your action:** run unchanged. **Output:** none expected; it defines the helper without sending requests yet. Request settings are logged without the key.


In [3]:
# @title 3. Prepare requests { display-mode: "form" }
RAW_RESPONSES = []
REQUEST_LOG = []

def api_get(endpoint, parameters):
    if OFFLINE:
        return offline_payload(endpoint, parameters)
    if endpoint not in {"search", "videos", "channels", "playlistItems", "commentThreads"}:
        raise ValueError("This workshop exposes only read endpoints.")
    import requests
    safe_parameters = {k: v for k, v in parameters.items() if k != "key"}
    parameters = dict(safe_parameters, key=API_KEY)
    for attempt in range(3):
        requested_at = datetime.now(timezone.utc).isoformat()
        try:
            response = requests.get(
                f"https://www.googleapis.com/youtube/v3/{endpoint}",
                params=parameters, timeout=(5, 25),
            )
        except requests.RequestException as exc:
            raise RuntimeError(f"Network request failed ({type(exc).__name__}); credentials omitted.") from None
        REQUEST_LOG.append({"endpoint": endpoint, "parameters": safe_parameters,
                            "status": response.status_code,
                            "requested_at": requested_at})
        if response.status_code in {429, 500, 502, 503, 504} and attempt < 2:
            retry_after = response.headers.get("Retry-After", "")
            delay = min(float(retry_after), 30) if retry_after.isdigit() else 2 ** attempt
            time.sleep(delay)
            continue
        try:
            payload = response.json()
        except ValueError:
            raise RuntimeError(f"HTTP {response.status_code}: response was not valid JSON.") from None
        if not isinstance(payload, dict):
            raise RuntimeError("Unexpected API response: expected a JSON object.")
        if response.status_code != 200 or "error" in payload:
            reasons = [str(e.get("reason", "unknown"))
                       for e in payload.get("error", {}).get("errors", [])]
            raise RuntimeError(f"API request stopped: HTTP {response.status_code}, reasons={reasons}")
        if not isinstance(payload.get("items"), list):
            raise RuntimeError("Unexpected API response: expected an items list.")
        RAW_RESPONSES.append({"endpoint": endpoint, "parameters": safe_parameters,
                              "received_at": datetime.now(timezone.utc).isoformat(),
                              "payload": payload})
        return payload
    raise RuntimeError("Request did not succeed within the retry budget.")


## 4. Identify the channel
In live mode, this cell looks up your chosen @handle and stores the returned channel ID in `CHANNEL_ID`. Later requests use that ID. In saved mode it selects the fixed saved example.

**Your action:** run unchanged. **Output:** the live channel name and ID, or a saved-mode message. A “Channel not found” error means check the handle in the first cell and rerun.


In [4]:
# @title 4. Identify the channel { display-mode: "form" }
if OFFLINE:
    print("Using the saved AfD example; original observation time unknown.")
else:
    channel = api_get("channels", {"part": "snippet", "forHandle": CHANNEL_HANDLE.strip()})
    if not channel["items"]:
        raise ValueError("Channel not found. Check its @handle.")
    CHANNEL_ID = channel["items"][0]["id"]
    print(channel["items"][0]["snippet"]["title"], CHANNEL_ID)


Using the saved AfD example; original observation time unknown.


## 5. Find videos and extract their IDs
This cell searches for videos from the selected channel using your ranking and page size. It extracts unique video IDs for the next request. Search results contain titles and IDs but not view counts.

**Your action:** run unchanged. **Output:** the number of results, a first-result example and the ID list. Saved mode ranks only the cached videos; it does not search YouTube.


In [5]:
# @title 5. Search for video IDs { display-mode: "form" }
search_params = {
    "part": "snippet", "channelId": CHANNEL_ID,
    "type": "video", "order": BASE_ORDER, "maxResults": PAGE_SIZE,
}
search_data = api_get("search", search_params)
print("Videos on this page:", len(search_data["items"]))
if search_data["items"]:
    first = search_data["items"][0]
    print(json.dumps({"id": first["id"], "snippet": {"title": first.get("snippet", {}).get("title")}},
                     indent=2, ensure_ascii=False))

video_ids = list(dict.fromkeys(item["id"]["videoId"] for item in search_data["items"]))
print(video_ids)


Videos on this page: 10
{
  "id": {
    "videoId": "8ioWqF4k2fo"
  },
  "snippet": {
    "title": "Brandmauer weg oder Neuwahlen! PK mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag"
  }
}
['8ioWqF4k2fo', 'eYh7yfnluqs', 'Uf9nmuTDesk', 'Hz6SLP23Nv0', 'OU-pD4Tm90w', 'HbPnlbTchM8', '5m7dIsDftmQ', 'IehYwnYBwZs', '40qjQvDzUII', 'iwycKryn3aw']


## 6. Request video details
This cell sends the IDs to the `videos` endpoint and requests descriptions, statistics and content details. View counts belong to individual videos here, rather than channel totals such as subscribers.

**Your action:** run unchanged and find `statistics.viewCount` in the output. **Output:** one video’s details and a list of IDs without returned metadata. `[]` means none are missing.


In [6]:
# @title 6. Get metadata { display-mode: "form" }
stats_params = {"part": "snippet,statistics,contentDetails", "id": ",".join(video_ids)}
stats_data = api_get("videos", stats_params) if video_ids else {"items": []}
if stats_data["items"]:
    first = stats_data["items"][0]
    print(json.dumps({"id": first["id"], "snippet": {"title": first.get("snippet", {}).get("title")},
                      "statistics": first.get("statistics", {})}, indent=2, ensure_ascii=False))
returned_ids = {item["id"] for item in stats_data["items"]}
print("IDs without metadata:", sorted(set(video_ids) - returned_ids))


{
  "id": "8ioWqF4k2fo",
  "snippet": {
    "title": "Brandmauer weg oder Neuwahlen! PK mit Alice Weidel & Tino Chrupalla - AfD-Fraktion im Bundestag"
  },
  "statistics": {
    "viewCount": "308065",
    "likeCount": "25574",
    "commentCount": "2288"
  }
}
IDs without metadata: []


## 7. Build a table
This cell selects fields from the nested response and converts counts into numbers and dates into date-time values. Missing counts stay missing rather than becoming zero.

**Your action:** run unchanged. **Output:** `baseline_df`, with one row per returned video. The preview shows IDs, titles, views and comment counts; the full table also includes dates, likes, duration and channel IDs.


In [7]:
# @title 7. Build the table { display-mode: "form" }
def make_table(payload):
    rows = [{
        "video_id": item["id"],
        "title": item.get("snippet", {}).get("title"),
        "published_at": item.get("snippet", {}).get("publishedAt"),
        "channel_id": item.get("snippet", {}).get("channelId"),
        "duration": item.get("contentDetails", {}).get("duration"),
        "views": item.get("statistics", {}).get("viewCount"),
        "likes": item.get("statistics", {}).get("likeCount"),
        "comments": item.get("statistics", {}).get("commentCount"),
    } for item in payload["items"]]
    columns = ["video_id", "title", "published_at", "channel_id", "duration", "views", "likes", "comments"]
    frame = pd.DataFrame(rows, columns=columns)
    for field in ["views", "likes", "comments"]:
        frame[field] = pd.to_numeric(frame[field], errors="raise").astype("Int64")
    frame["published_at"] = pd.to_datetime(frame["published_at"], utc=True)
    return frame

baseline_df = make_table(stats_data)
print(baseline_df[["video_id", "title", "views", "comments"]].to_string(index=False, max_colwidth=45))


   video_id                                         title  views  comments
8ioWqF4k2fo Brandmauer weg oder Neuwahlen! PK mit Alic... 308065      2288
eYh7yfnluqs Interview mit Beatrix von Storch zum Ergeb... 249565      1563
Uf9nmuTDesk Regierungsbefragung mit Lars Klingbeil und... 149674      1822
Hz6SLP23Nv0 LIVE: Pressekonferenz der AfD-Fraktion - D... 147981      1183
OU-pD4Tm90w BUNDESTAG LIVE - 80. Sitzung - AfD-Fraktio... 117253         5
HbPnlbTchM8 Dr. Anna Rathert zum Untersuchungsausschus... 112037       805
5m7dIsDftmQ Bernd Baumann knöpft sich die Umfrageverli... 103756       299
IehYwnYBwZs BUNDESTAG LIVE - 79. Sitzung - AfD-Fraktio...  98033        25
40qjQvDzUII Nordstream: Wir fordern einen Untersuchung...  81091       695
iwycKryn3aw LIVE: Presseerklärung mit Alice Weidel & T...  72894       612


## 8. Read one result
The next cell summarises views, likes and comment counts for the returned videos. `count` is the number of non-missing values, `mean` the average, and `median` the middle value. `min` and `max` show the range.

**Your action:** run unchanged. Compare mean and median views and look in the table for videos driving the difference.

**Output:** a summary table. These are selected records, not a representative sample of YouTube; saved counts are historical.


In [8]:
# @title 8. Descriptive statistics { display-mode: "form" }
numeric_data = baseline_df[["views", "likes", "comments"]].dropna(axis=1, how="all")
if not numeric_data.empty:
    print(numeric_data.agg(["count", "mean", "median", "min", "max"]).T)


          count      mean    median      min       max
views      10.0  144034.9  114645.0  72894.0  308065.0
likes      10.0    9825.6    7669.5   2224.0   25574.0
comments   10.0     929.7     750.0      5.0    2288.0


## 9. Save your data and settings
This cell creates a timestamped run folder, saves the video table as `worked_example.csv`, and writes `manifest.json` with the mode, channel, ranking, page size and request settings.

**Your action:** run unchanged. **Output:** the CSV’s path. The manifest helps you explain how the records were selected; it omits your API key. In saved mode, the original observation time remains unknown.


In [9]:
# @title 9. Save CSV and collection settings { display-mode: "form" }
run_dir = OUTPUT_DIR / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
baseline_df.to_csv(run_dir / "worked_example.csv", index=False)
manifest = {"mode": "saved_example" if OFFLINE else "live", "channel_id": CHANNEL_ID,
            "ordering": BASE_ORDER, "page_size": PAGE_SIZE,
            "original_observation_time": None if OFFLINE else [r["received_at"] for r in RAW_RESPONSES],
            "requests": REQUEST_LOG}
(run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", run_dir / "worked_example.csv")


Saved: workshop_outputs/20261106T061545818125Z/worked_example.csv


Download the CSV and manifest from the printed folder inside `workshop_outputs/` in Colab’s left file browser.

Optional summary: **your question, request choices, one result, one limitation**. For example: “Among these selected videos, which has the most views?”

[Attribution and saved-data provenance](https://github.com/manueltonneau/hpi-api/blob/main/SOURCES.md)
